In [ ]:
import os
import csv
import numpy as np
import nibabel as nib
import pandas as pd


#--------------------------------------
# change the path !! 
#--------------------------------------
INPUT_CSV = '/path/to/MELD-PostOp/model_performance/demographics.csv' # incomplete csv file
OUTPUT_CSV = '/path/to/MELD-PostOp/model_performance/analysis.csv' # complete csv file to use for next step
GT_DIR = '/path/to/MELD-PostOp/ground_truth'
PRED_DIR = '/path/to/MELD-PostOp/output'
POSTPOP_DIR = '/path/to/MELD-PostOp/input'

In [ ]:
def dice_score(mask1, mask2):
    mask1 = mask1.astype(bool)
    mask2 = mask2.astype(bool)
    denom = mask1.sum() + mask2.sum()
    if denom == 0:
        return np.nan
    return 2.0 * np.logical_and(mask1, mask2).sum() / denom


def calculate_volume(mask_img):
    data = mask_img.get_fdata() > 0
    voxel_vol = np.prod(mask_img.header.get_zooms())
    return data.sum() * voxel_vol / 1000.0 


def isotropy(file_path, tolerance=1e-3):
    try:
        nii_img = nib.load(file_path)
        affine = nii_img.affine

        voxel_sizes = np.sqrt(np.sum(affine[:3, :3]**2, axis=0))
        is_isotropic = np.all(np.abs(voxel_sizes - voxel_sizes[0]) < tolerance)
    
        
        results = {
            'isotropy_status': 'isotropic' if is_isotropic else 'anisotropic',
        }
        return results
        
    except Exception as e:
        return {'file_path': file_path, 'error': f"Error processing {file_path}: {str(e)}"}


df = pd.read_csv(INPUT_CSV)

# Ensure columns exist
if "DSC" not in df.columns:
    df["DSC"] = np.nan
if "manual_volume" not in df.columns:
    df["manual_volume"] = np.nan
if "image_isotropy" not in df.columns:
    df["image_isotropy"] = np.nan

In [ ]:
# Main loop

for idx, row in df.iterrows():

    sub = row["id"]
    mri_path = os.path.join(POSTPOP_DIR, f"sub-{sub}_0000.nii.gz")
    gt_path = os.path.join(GT_DIR, f"sub-{sub}_gt.nii.gz")
    pred_path = os.path.join(PRED_DIR, f"sub-{sub}.nii.gz")

    if not os.path.exists(gt_path) or not os.path.exists(pred_path) or not os.path.exists(mri_path):
        print(f"Missing files for {sub}, skipping.")
        continue


    gt_img = nib.load(gt_path)
    pred_img = nib.load(pred_path)

   
    gt_data = gt_img.get_fdata() > 0
    pred_data = pred_img.get_fdata() > 0

    dsc = dice_score(gt_data, pred_data)
    vol = calculate_volume(gt_img)
    iso = isotropy(mri_path)

    df.at[idx, "DSC"] = dsc
    df.at[idx, "manual_volume"] = vol
    df.at[idx, "image_isotropy"] = iso["isotropy_status"]

    print(f"{sub}: DSC={dsc:.4f}, volume={vol:.2f} cm^3")


# Save updated CSV
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved updated CSV to:\n{OUTPUT_CSV}")